# A Piece-wise Model for Vanilla Option Pricing

In this notebook, we provide a standalone Python implementation of the piece-wise model for Vanilla option pricing discussed in (TBD).

The notebook covers methods to setup the model and calculate prices and implied volatilites.

For model calibration methods, we refer to the Julia implementation and the example in [JaeckelExample.ipynb](../notebooks/JaeckelExample.ipynb).

The implementation and examples use standard Python packages.

In [ ]:
from scipy.stats import norm
from scipy.optimize import brentq
import numpy as np
import pandas as pd

import plotly.graph_objects as go

## Auxiliary Functions for Option Prices and Implied Volatilities

In [ ]:
def black(strike, forward, sigma, T, callOrPut):
    nu = sigma*np.sqrt(T)
    if np.abs(nu)<1.0e-12:   # assume zero
        return max(callOrPut*(forward-strike),0.0)  # intrinsic value
    moneyness = forward/strike
    d1 = np.log(moneyness) / nu + nu / 2.0
    d2 = d1 - nu
    return callOrPut * strike * (moneyness*norm.cdf(callOrPut*d1)-norm.cdf(callOrPut*d2))

def black_implied_vol(price, strike, forward, T, callOrPut):
    def objective(sigma):
        return black(strike, forward, sigma, T, callOrPut) - price
    return brentq(objective,0.01, 1.00, xtol=1.0e-8)

def bachelier(strike, forward, sigma, T, callOrPut):
    moneyness = forward-strike
    nu = sigma * np.sqrt(T)
    h = callOrPut * moneyness / nu
    return nu * (h*norm.cdf(h) + norm.pdf(h))

def bachelier_implied_vol(price, strike, forward, T, callOrPut):
    def objective(sigma):
        return bachelier(strike, forward, sigma, T, callOrPut) - price
    return brentq(objective,1e-4, 1e-1, xtol=1.0e-8)


## Piece-wise Model Implementation

The implementation is split into low-level functions that facilitate relevant calculations and a class which provides high-level user methods.

In [ ]:
const_r_eps = 1.0e-8  # r -> 0, avoid division by zero

In [ ]:
def brownian_grid(v0, ds, dv):
    assert len(ds) == len(dv)
    dw = np.zeros(ds.shape)
    for k in range(dw.shape[0]):
        ds0_ = 0.0 if k==0 else ds[k-1]
        dv0_ = 0.0 if k==0 else dv[k-1]
        dw0_ = 0.0 if k==0 else dw[k-1]
        #
        ds1_ = ds[k]
        dv1_ = dv[k]
        #
        r1 = (dv1_ - dv0_) / (ds1_ - ds0_)
        if np.abs(r1) < const_r_eps:
            step = 1.0 / (v0 + dv0_)
        else:
            step = (np.log(v0 + dv1_) - np.log(v0 + dv0_)) / (dv1_ - dv0_)
        dw[k] = dw0_ + (ds1_ - ds0_) * step
    return dw


In [ ]:
def initial_values(idx, v0, ds, dv, dw, r_extrapolation):
    if idx < 0:
        dw0 = 0.0
        ds0 = 0.0
        dv_ = 0.0
        r_ = dv[0] / ds[0]
    else:
        dw0 = dw[idx]
        ds0 = ds[idx]
        dv_ = dv[idx]
        if idx < len(dv)-1:
            r_ = (dv[idx+1] - dv[idx]) / (ds[idx+1] - ds[idx])
        else:
            if r_extrapolation is not None:
                r_ = r_extrapolation
            else:
                r_ = initial_values(idx-1, v0, ds, dv, dw, r_extrapolation)[3]
    return (dw0, ds0, v0 + dv_, r_)

In [ ]:
def risk_factor(dw_, v0, ds, dv, dw, r_extrapolation):
    assert dw_ >= 0.0
    assert len(ds) == len(dv)
    assert len(ds) == len(dw)
    idx = np.searchsorted(dw, dw_, 'right') - 1
    (dw0, ds0, v_, r_) = initial_values(idx, v0, ds, dv, dw, r_extrapolation)
    if np.abs(r_) < const_r_eps:
        step = dw_ - dw0
    else:
        step = (np.exp(r_*(dw_-dw0)) - 1.0) / r_
    ds_ = ds0 + v_ * step
    return ds_

In [ ]:
def brownian_factor(ds_, v0, ds, dv, dw, r_extrapolation):
    assert ds_ >= 0.0
    assert len(ds) == len(dv)
    assert len(ds) == len(dw)
    idx = np.searchsorted(ds, ds_, 'right') - 1
    (dw0, ds0, v_, r_) = initial_values(idx, v0, ds, dv, dw, r_extrapolation)
    if np.abs(r_) < const_r_eps:
        step = 1.0/v_ * (ds_ - ds0)
    else:
        tmp = 1.0 + r_/v_*(ds_ - ds0)
        if tmp <= 0.0:
            step = np.inf
        else:
            step = np.log(tmp) / r_
    dw_ = dw0 + step
    return dw_

In [ ]:
def local_volatility(ds_, v0, ds, dv, dw, r_extrapolation):
    assert ds_ >= 0.0
    assert len(ds) == len(dv)
    assert len(ds) == len(dw)
    idx = np.searchsorted(ds, ds_, 'right') - 1
    (dw0, ds0, v_, r_) = initial_values(idx, v0, ds, dv, dw, r_extrapolation)
    vol = v_ + r_ * (ds_ - ds0)
    return vol


In [ ]:
def call_spread_segment(strike, s0, s1, w0, v_, r_, T):
    assert strike <= s1
    assert s0 < s1
    lambda_ = 0.0
    if np.abs(r_) < const_r_eps:
        F = s0 - v_ * w0
        C0 = bachelier(np.maximum(strike, s0), F, v_, T, 1.0)
        if s1 == np.inf:
            C1 = 0.0
        else:
            C1 = bachelier(s1, F, v_, T, 1.0)
    else:
        lambda_ = v_ / r_ - s0
        F = (s0 + lambda_) * np.exp(r_*(r_*T/2.0 - w0)) - lambda_
        C0 = black(np.maximum(strike, s0) + lambda_, F + lambda_, r_, T, 1.0)
        if s1 == np.inf:
            C1 = 0.0
        else:
            C1 = black(s1 + lambda_, F + lambda_, r_, T, 1.0)
    #
    # print(F, ', ', s0, ', ', s1, ', ', r_, ', ', lambda_)
    return (C0 - C1)

In [ ]:
def put_spread_segment(strike, s0, s1, w0, v_, r_, T):
    assert strike >= s1
    assert s1 < s0
    lambda_ = 0.0
    if np.abs(r_) < const_r_eps:
        F = s0 - v_ * w0
        P0 = bachelier(np.minimum(strike, s0), F, v_, T, -1.0)
        if s1 == -np.inf:
            P1 = 0.0
        else:
            P1 = bachelier(s1, F, v_, T, -1.0)
    else:
        lambda_ = v_ / r_ - s0
        F = (s0 + lambda_) * np.exp(r_*(r_*T/2.0 - w0)) - lambda_
        P0 = black(np.minimum(strike, s0) + lambda_, F + lambda_, r_, T, -1.0)
        if s1 == -np.inf:
            P1 = 0.0
        else:
            P1 = black(s1 + lambda_, F + lambda_, r_, T, -1.0)
    #
    # print(F, ', ', s0, ', ', s1, ', ', r_, ', ', lambda_)
    return (P0 - P1)

In [ ]:
def call_option(strike, s0, v0, w0, ds, dv, dw, r_extrapolation, T):
    assert strike >= s0
    ds_ = strike - s0
    idx = np.searchsorted(ds, ds_, 'right') - 1
    C = 0.0
    for k in range(idx, len(ds)):
        (dw0, ds0, v_, r_) = initial_values(k, v0, ds, dv, dw, r_extrapolation)
        w_k = w0 + dw0
        s_k = s0 + ds0
        if k < len(ds)-1:
            s_k_p_1 = s0 + ds[k+1]
        else:
            s_k_p_1 = np.inf
        call_seg = call_spread_segment(strike, s_k, s_k_p_1, w_k, v_, r_, T)
        C += call_seg
    return C # np.maximum(C, 0.0)

In [ ]:
def put_option(strike, s0, v0, w0, ds, dv, dw, r_extrapolation, T):
    assert strike <= s0
    ds_ = s0 - strike
    idx = np.searchsorted(ds, ds_, 'right') - 1
    P = 0.0
    for k in range(idx, len(ds)):
        (dw0, ds0, v_, r_) = initial_values(k, v0, ds, dv, dw, r_extrapolation)
        w_k = w0 - dw0
        s_k = s0 - ds0
        if k < len(ds)-1:
            s_k_p_1 = s0 - ds[k+1]
        else:
            s_k_p_1 = -np.inf
        put_seg = put_spread_segment(strike, s_k, s_k_p_1, w_k, v_, -r_, T)  # beware the sign
        P += put_seg
    return P # np.maximum(P, 0.0)

In [ ]:
class PieceWiseModel:

    def __init__(self, *args, **kwargs):
        self.initialse(*args, **kwargs)


    def initialse(self, s0, v0, w0, T, dsl, dsu, dvl, dvu, rexl = 0.0, rexu = 0.0):
        assert v0 > 0.0
        assert T > 0.0
        assert dsl.shape == dvl.shape
        assert dsu.shape == dvu.shape
        self.s0 = s0
        self.v0 = v0
        self.w0 = w0
        self.T = T
        self.dsl = dsl
        self.dsu = dsu
        self.dvl = dvl
        self.dvu = dvu
        self.rexl = rexl
        self.rexu = rexu
        #
        self.dwl = brownian_grid(v0, dsl, dvl)
        self.dwu = brownian_grid(v0, dsu, dvu)

    def risk_factor(self, w):
        if w >= self.w0:
            dw = w - self.w0
            ds = risk_factor(dw, self.v0, self.dsu, self.dvu, self.dwu, self.rexu)
            return self.s0 + ds
        else:
            dw = self.w0 - w
            ds = risk_factor(dw, self.v0, self.dsl, self.dvl, self.dwl, self.rexl)
            return self.s0 - ds

    def brownian_factor(self, s):
        if s >= self.s0:
            ds = s - self.s0
            dw = brownian_factor(ds, self.v0, self.dsu, self.dvu, self.dwu, self.rexu)
            return self.w0 + dw
        else:
            ds = self.s0 - s
            dw = brownian_factor(ds, self.v0, self.dsl, self.dvl, self.dwl, self.rexl)
            return self.w0 - dw

    def local_volatility(self, s):
        if s >= self.s0:
            ds = s - self.s0
            lv = local_volatility(ds, self.v0, self.dsu, self.dvu, self.dwu, self.rexu)
            return lv
        else:
            ds = self.s0 - s
            lv = local_volatility(ds, self.v0, self.dsl, self.dvl, self.dwl, self.rexl)
            return lv

    def implied_density(self, s):
        w = self.brownian_factor(s)
        v = self.local_volatility(s)
        d = norm.pdf(w) / v
        return d
    
    def vanilla_option(self, strike, call_or_put):
        if call_or_put == 1:
            assert strike >= self.s0  # add in-the-money options later
            return call_option(strike, self.s0, self.v0, self.w0, self.dsu, self.dvu, self.dwu, self.rexu, self.T)
        if call_or_put == -1:
            assert strike <= self.s0  # add in-the-money options later
            return put_option(strike, self.s0, self.v0, self.w0, self.dsl, self.dvl, self.dwl, self.rexl, self.T)
        #
        return 0.0

    def implied_log_volatility(self, strike):
        cp = 1.0 if strike >= self.s0 else -1
        o = self.vanilla_option(strike, cp)
        return black_implied_vol(o, strike, self.s0, self.T, cp)


## Model Testing

We test the model by setting up (boundary case) shifted log-normal and normal models.

### Normal Model

In [ ]:
dsl = np.array([ 0.2, 0.5 ])
dsu = np.array([ 0.5, 1.0, 2.0 ])
dvl = np.zeros(dsl.shape)
dvu = np.zeros(dsu.shape)

s0 = 1.0
v0 = 0.30
w0 = 0.0
T  = 3.0

m = PieceWiseModel(s0, v0, w0, T, dsl, dsu, dvl, dvu, rexl = 0.0, rexu = 0.0)

strikes = np.linspace(0.0, 1.0, 11)
put_tst = np.array([ m.vanilla_option(s, -1) for s in strikes ])
put_ref = np.array([bachelier(s, s0, v0, T, -1) for s in strikes ])
deviation = np.max(np.abs(put_tst - put_ref))
print('Deviation normal model puts:  ', deviation)

strikes = np.linspace(1.0, 3.0, 11)
cal_tst = np.array([ m.vanilla_option(s, 1) for s in strikes ])
cal_ref = np.array([bachelier(s, s0, v0, T, 1) for s in strikes ])
deviation = np.max(np.abs(cal_tst - cal_ref))
print('Deviation normal model calls: ', deviation)

### Lognormal Model

In [ ]:
dsl = np.array([ 0.2, 0.5 ])
dsu = np.array([ 0.5, 1.0, 2.0 ])

s0 = 1.0
v0 = 0.30
r  = 0.2
T  = 3.0

λ = v0/r - s0
w0 = 0.5*r*T
dvl = -r * dsl
dvu = r * dsu

m = PieceWiseModel(s0, v0, w0, T, dsl, dsu, dvl, dvu, rexl = None, rexu = None)

strikes = np.linspace(0.0, 1.0, 11)
put_tst = np.array([ m.vanilla_option(s, -1) for s in strikes ])
put_ref = np.array([black(s + λ, s0 + λ, r, T, -1) for s in strikes ])
deviation = np.max(np.abs(put_tst - put_ref))
print('Deviation normal model puts:  ', deviation)

strikes = np.linspace(1.0, 3.0, 11)
cal_tst = np.array([ m.vanilla_option(s, 1) for s in strikes ])
cal_ref = np.array([black(s + λ, s0 + λ, r, T, 1) for s in strikes ])
deviation = np.max(np.abs(cal_tst - cal_ref))
print('Deviation normal model calls: ', deviation)

## Example Model

We use an example smile analysed in (TBD) and setup the corresponding piece-wise model here.

For this step, we use the model parameters calibrated via the Julia implementation of the model.

In [ ]:
dsl = np.array([0.28458157363164205, 0.488176475212622, 0.633832019318307, 0.738036679474224, 0.81258661346322, 0.865921009923492, 0.904077419910406, 0.931375218699109, 0.950904566951844, 0.964876222546815])
dsu = np.array([0.39778339939641993, 0.9537984316282099, 1.73098701349666, 2.8173283114328393, 4.33579814376678, 6.458290067887431, 9.4250740447762, 13.571995437266702, 19.3684933182917, 27.4707418310251])
dvl = np.array([0.02184801483669285, 0.022255740067016822, 0.03394909335635351, 0.04828569325848149, 0.0654166998073218, 0.0800980191490987, 0.08701556567463874, 0.07575771272897543, 0.10030995216666, -0.03311227641329198])
dvu = np.array([0.0649173163905064, 0.18545544763267813, 0.3720896848930713, 0.6332283601234567, 0.9508680939486378, 1.3493722887301163, 1.9397457303377001, 2.8040498215059553, 4.075597582348989, 6.1460328046249515])

m = PieceWiseModel(
    s0 = 1.0,
    v0 = 0.2101274979752493,
    w0 = 0.20309627585111245,
    T = 5.0722,
    dsl = dsl,
    dsu = dsu,
    dvl = dvl,
    dvu = dvu,
    rexl = None,
    rexu = 0.2
)

As an initial test, we verify call-put parity for the model.

In [ ]:
c = m.vanilla_option(1.0, 1.0)
p = m.vanilla_option(1.0, -1.0)
print('ATM call: ', c)
print('ATM put:  ', p)

We illustrate a couple of model properties.

In [ ]:
log_strikes = np.linspace(-3.0, 3.0, 501)
strikes     = np.exp(log_strikes)

impl_vols = np.array([ m.implied_log_volatility(s) for s in strikes ])
local_vols = np.array([ m.local_volatility(s) for s in strikes ])
impl_dens = np.array([ m.implied_density(s) for s in strikes ])

In [ ]:
layout = go.Layout(title=dict(text='Example model - implied log-volatility'))
fig = go.Figure(
    data=go.Scatter(
        x=log_strikes,
        y=impl_vols * 1.e+2,
        mode='lines',
    ),
    layout=layout,
)
fig.update_xaxes(title_text = "Log-moneyness",)
fig.update_yaxes(title_text = "volatility (%)",)
fig.show()

In [ ]:
layout = go.Layout(title=dict(text='Example model - volatility parameters'))
fig = go.Figure(
    data=go.Scatter(
        x=log_strikes,
        y=local_vols * 1.e+2,
        mode='lines',
    ),
    layout=layout,
)
fig.update_xaxes(title_text = "Log-moneyness",)
fig.update_yaxes(title_text = "volatility (%)",)
fig.show()

In [ ]:
layout = go.Layout(title=dict(text='Example model - implied density'))
fig = go.Figure(
    data=go.Scatter(
        x=log_strikes,
        y=impl_dens,
        mode='lines',
    ),
    layout=layout,
)
fig.update_xaxes(title_text = "Log-moneyness",)
fig.update_yaxes(title_text = "density", type='log')
fig.show()

## Comparison with Julia Implementation

To double-check our implementation, we compare the Python implementation against the independent Julia implementation.

Note that the Julia code implements the call/put-spread differently avoiding a few cumulative normal distribution function calls. Thus this is indeed a consistency check of two implementation approaches.

In [ ]:
tst_strikes = [ 0.05, 0.1, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 5.0, 10.0]
ref_vols = [ 0.6202035273714095, 0.548149455436994, 0.33066762129926885, 0.27950100007826045, 0.24932888288165445, 0.23406732950836334, 0.22648619502073036, 0.22057221673846739, 0.2185603690200145, 0.2159104513104457,]
tst_vols = [ m.implied_log_volatility(s) for s in tst_strikes ]

pd.DataFrame([tst_strikes, ref_vols, tst_vols], index=['Strike', 'RefVol', 'TstVol'])

We see small deviations between test (Python) and reference (Julia) volatilities for very low and very high strikes. These deviations are attributed to differences when calculating normal CDFs at very high or low levels.